# Ovarian OS first pass — clinical baseline, expression + clinical, LASSO-Cox, RSF, DeepSurv

Trains on the compiled RNA-seq + Affymetrix pool (`os-training-pool/`, 751 / 485)
and scores **all five** held-out validators (`os-validation/`, 892 / 410). Follows
`research/os-hgsoc/workstream.md` (modelling protocol); tracked as experiment **E003**.

| Class | What |
|---|---|
| Clinical Cox | residual + stage + age **where present**; no imputation. Transported: residual + stage fit on the pool (stratified by cohort) |
| **Expression + clinical Cox (primary, D009)** | two-stage: LASSO-Cox expression score + residual + stage, fit on the pool stratified by cohort |
| LASSO-Cox | L1 Cox on rank-transformed expression (`sksurv` Coxnet, coordinate descent) |
| Random survival forest | `sksurv.ensemble.RandomSurvivalForest` |
| DeepSurv | one small MLP, Cox partial-likelihood loss |

Preprocessing (modelling-time, not baked into the CSVs):

- Residual / FIGO recoded by `agent/scripts/os_clinical.py` (GOG ≤1 cm optimal).
- Expression is **not** ComBat-corrected. Each sample is rank-transformed (percentile
  across genes) so RNA-seq and microarray share a rank scale, then the top 500
  genes by training variance are z-scored on the pool.
- Validation empty cells (~2.8% platform gaps) become mid-rank (0.5) before z-scoring.

**Primary comparison (pre-registered, D006/D009):** `expr_clin_cox` vs `clinical_transported`,
ΔC ≥ 0.03 on ≥ 2 validators. Gene-only models are secondary.

**Dry run:** a run not linked to E003 (no `--experiment E003`) replaces validation outcomes
with synthetic noise, skips `register_validation`, and saves nothing — so the whole pipeline
can be tested without spending the single scoring the ledger allows.

Internal OOF numbers are **not** evidence. The claim is external C-index (bootstrap
CI), KM log-rank, and ΔC vs the clinical baseline, per validator.

**Running it** (the agent does this itself; see the colab-compute skill)

```bash
.venv/bin/python agent/scripts/colab_sync.py start --dataset os-training-pool --dataset os-validation
.venv/bin/python agent/scripts/colab_sync.py run --experiment E003 notebooks/ovarian-os-first-pass.ipynb
.venv/bin/python agent/scripts/colab_sync.py stop
```

`run` executes every cell on the Colab session, writes the executed copy to
`ovarian-os-first-pass_output.ipynb`, and imports the run record into
`.research/colab/runs/`. CPU is enough; a GPU only speeds up DeepSurv.

## 1. Bootstrap

Finds the workspace — the copy `colab_sync.py` uploads to `/content/research`
on a Colab session, or the local checkout on a local kernel — and imports
`colab_env`. Nothing is cloned.

In [ ]:
import importlib
import pathlib
import sys

RUNTIME_DIR = pathlib.Path("/content/research")  # .research/config.yaml -> colab.runtime_dir
candidates = [RUNTIME_DIR, pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
root = next((p for p in candidates if (p / "agent" / "scripts" / "colab_env.py").exists()), None)
if root is None:
    raise SystemExit("workspace not found: run `agent/scripts/colab_sync.py start` locally first")

sys.path.insert(0, str(root / "agent" / "scripts"))
import colab_env as ce
importlib.reload(ce)
print(ce.summary())

## 2. Dependencies

Colab already has pandas / sklearn / matplotlib / torch. `lifelines` and
`scikit-survival` come from `agent/requirements-colab.txt`.


In [ ]:
print(ce.require("pandas", "numpy", "scikit-learn", "matplotlib", "lifelines",
                 "torch", "scikit-survival"))
print(ce.install_requirements())
import os_clinical
importlib.reload(os_clinical)
print("os_clinical recoding loaded")


## 3. Load compiled tables and recode

Fetch `os-training-pool` and `os-validation` (slug-root layout, not `csv/`).
Assert the join rule: expression columns = labels `sample_id` order.


In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from lifelines import CoxPHFitter, KaplanMeierFitter
from lifelines.statistics import logrank_test
from lifelines.utils import concordance_index
from scipy.stats import chi2

N_GENES = 500
N_BOOT = 300
SEED = 0
VAL_COHORTS = ["gse32062", "gse53963", "gse17260", "gse140082", "gse49997"]
CLIN_COLS = ["residual_subopt", "stage_ord", "age"]

paths = ce.ensure_os_tables()
print({k: str(v) for k, v in paths.items()})


def load_split(root, expr_name):
    lab = pd.read_csv(root / "labels.csv")
    lab["sample_id"] = lab["sample_id"].astype(str)
    lab = lab.set_index("sample_id")
    expr = pd.read_csv(root / expr_name, index_col=0)
    expr.columns = expr.columns.astype(str)
    if list(expr.columns) != list(lab.index):
        raise SystemExit("expression columns do not match labels sample_id order")
    lab["residual_subopt"] = lab["residual_disease"].map(os_clinical.recode_residual)
    lab["stage_ord"] = lab["figo_stage"].map(os_clinical.recode_stage)
    lab["age"] = pd.to_numeric(lab["age_years"], errors="coerce")
    lab["os_time_days"] = pd.to_numeric(lab["os_time_days"], errors="coerce")
    lab["os_event"] = pd.to_numeric(lab["os_event"], errors="coerce").astype(int)
    return lab, expr.T.astype(np.float32)


lab_tr, X_tr = load_split(paths["train"], "expression_pool.csv")
lab_va, X_va = load_split(paths["val"], "expression_validation.csv")
time_tr, event_tr = lab_tr["os_time_days"], lab_tr["os_event"]

# Only a run linked to E003 sees real validation outcomes. Anything else (a smoke run without
# --experiment) is a dry run: validation outcomes become synthetic noise, the candidate is not
# registered, and nothing is saved.
EXPERIMENT = ce.run_context().get("experiment")
DRY_RUN = EXPERIMENT != "E003"
if DRY_RUN:
    fake = np.random.default_rng(12345)
    lab_va["os_time_days"] = np.ceil(fake.exponential(1500.0, len(lab_va)))
    lab_va["os_event"] = fake.binomial(1, 0.5, len(lab_va)).astype(int)
    print("DRY RUN (not linked to E003): validation outcomes are synthetic; nothing is registered or saved")

print(f"train {X_tr.shape}  events={int(event_tr.sum())}/{len(event_tr)}  "
      f"expr NA={int(X_tr.isna().sum().sum())}")
print(f"val   {X_va.shape}  events={int(lab_va['os_event'].sum())}/{len(lab_va)}  "
      f"expr NA={int(X_va.isna().sum().sum())}")

print("\nrecode completeness (non-null / n)")
for name, lab in ("train", lab_tr), ("val", lab_va):
    print(f"  {name}")
    for cohort, g in lab.groupby("cohort"):
        bits = [f"{c}={int(g[c].notna().sum())}/{len(g)}" for c in CLIN_COLS]
        print(f"    {cohort:16s} n={len(g):3d}  " + "  ".join(bits))

## 4. Expression ranks and the 500-gene matrix

Within-sample percentile ranks (no ComBat). Variance filter **on the training
pool only**, then z-score with training mean/sd. Validation NAs → mid-rank 0.5.


In [ ]:
X_tr_rank = X_tr.rank(axis=1, pct=True, method="average")
var = X_tr_rank.var(axis=0, ddof=0)
keep = var.nlargest(N_GENES).index
print(f"kept {len(keep)} genes  variance range {float(var[keep].min()):.5f}–{float(var[keep].max()):.5f}")

scaler = StandardScaler()
Z_tr = pd.DataFrame(
    scaler.fit_transform(X_tr_rank[keep]),
    index=X_tr.index, columns=keep, dtype=np.float32,
)

X_va_rank = X_va.rank(axis=1, pct=True, method="average").fillna(0.5)
Z_va = pd.DataFrame(
    scaler.transform(X_va_rank[keep]),
    index=X_va.index, columns=keep, dtype=np.float32,
)
print(f"Z_tr {Z_tr.shape}  Z_va {Z_va.shape}  val NA after fill {int(Z_va.isna().sum().sum())}")


## 5. Metrics helpers


In [ ]:
def cindex(time, event, risk):
    t = np.asarray(time, float)
    e = np.asarray(event, int)
    r = np.asarray(risk, float)
    ok = np.isfinite(t) & np.isfinite(r) & np.isfinite(e)
    if ok.sum() < 10 or e[ok].sum() < 2:
        return float("nan")
    return float(concordance_index(t[ok], -r[ok], e[ok]))


def cindex_ci(time, event, risk, n_boot=N_BOOT, seed=SEED):
    t = np.asarray(time, float)
    e = np.asarray(event, int)
    r = np.asarray(risk, float)
    ok = np.isfinite(t) & np.isfinite(r) & np.isfinite(e)
    t, e, r = t[ok], e[ok], r[ok]
    point = cindex(t, e, r)
    n = len(t)
    stats = []
    gen = np.random.default_rng(seed)
    for _ in range(n_boot):
        ix = gen.integers(0, n, n)
        if e[ix].sum() < 2:
            continue
        try:
            stats.append(concordance_index(t[ix], -r[ix], e[ix]))
        except Exception:
            continue
    if len(stats) < 20:
        return point, None, None
    lo, hi = np.percentile(stats, [2.5, 97.5])
    return point, float(lo), float(hi)


def km_logrank(time, event, risk):
    t = np.asarray(time, float)
    e = np.asarray(event, int)
    r = np.asarray(risk, float)
    ok = np.isfinite(t) & np.isfinite(r) & np.isfinite(e)
    t, e, r = t[ok], e[ok], r[ok]
    hi = r >= np.median(r)
    if hi.sum() < 5 or (~hi).sum() < 5:
        return None
    return float(logrank_test(t[hi], t[~hi], e[hi], e[~hi]).p_value)


def pack_metrics(time, event, risk):
    ci, lo, hi = cindex_ci(time, event, risk)
    r = np.asarray(risk, float)
    e = np.asarray(event, int)
    ok = np.isfinite(r)
    return {
        "n": int(ok.sum()),
        "events": int(e[ok].sum()) if ok.any() else 0,
        "cindex": ci,
        "cindex_ci_lo": lo,
        "cindex_ci_hi": hi,
        "logrank_p": km_logrank(time, event, risk),
    }


def linear_predictor(cph, features):
    # Uncentred Cox linear predictor x·beta. Higher = higher risk; comparable across fits.
    cols = list(cph.params_.index)
    return features[cols].astype(float).to_numpy() @ cph.params_.to_numpy()


def _boot_c(t, e, r):
    try:
        return concordance_index(t, -r, e)
    except ZeroDivisionError:
        return np.nan


def delta_ci(time, event, risk_a, risk_b, n_boot=N_BOOT, seed=SEED):
    # Paired bootstrap of C(risk_a) - C(risk_b) on the same patients -> point, lo, hi, se.
    t = np.asarray(time, float)
    e = np.asarray(event, int)
    a = np.asarray(risk_a, float)
    b = np.asarray(risk_b, float)
    ok = np.isfinite(t) & np.isfinite(e) & np.isfinite(a) & np.isfinite(b)
    t, e, a, b = t[ok], e[ok], a[ok], b[ok]
    point = cindex(t, e, a) - cindex(t, e, b)
    if not np.isfinite(point):
        return None, None, None, None
    stats = []
    gen = np.random.default_rng(seed)
    for _ in range(n_boot):
        ix = gen.integers(0, len(t), len(t))
        if e[ix].sum() < 2:
            continue
        d = _boot_c(t[ix], e[ix], a[ix]) - _boot_c(t[ix], e[ix], b[ix])
        if np.isfinite(d):
            stats.append(d)
    if len(stats) < 20:
        return float(point), None, None, None
    lo, hi = np.percentile(stats, [2.5, 97.5])
    return float(point), float(lo), float(hi), float(np.std(stats, ddof=1))


def random_effects(estimates, ses):
    # DerSimonian-Laird pooled estimate across validators, 95% CI, tau², I².
    y = np.asarray([np.nan if v is None else v for v in estimates], float)
    s = np.asarray([np.nan if v is None else v for v in ses], float)
    ok = np.isfinite(y) & np.isfinite(s) & (s > 0)
    y, s = y[ok], s[ok]
    if len(y) < 2:
        return None
    w = 1.0 / s**2
    fixed = np.sum(w * y) / np.sum(w)
    q = float(np.sum(w * (y - fixed) ** 2))
    df = len(y) - 1
    c = np.sum(w) - np.sum(w**2) / np.sum(w)
    tau2 = max(0.0, (q - df) / c) if c > 0 else 0.0
    wr = 1.0 / (s**2 + tau2)
    est = float(np.sum(wr * y) / np.sum(wr))
    se = float(np.sqrt(1.0 / np.sum(wr)))
    return {"estimate": est, "ci_lo": est - 1.96 * se, "ci_hi": est + 1.96 * se, "se": se,
            "tau2": float(tau2), "i2": max(0.0, (q - df) / q) if q > 0 else 0.0, "k": int(len(y))}


def stratified_cindex(lab, risk):
    # Event-weighted mean of per-cohort C-indices (pooled cohorts have different baseline hazards).
    num = den = 0.0
    for _, g in lab.groupby("cohort"):
        c = cindex(g["os_time_days"], g["os_event"], risk.loc[g.index])
        if np.isfinite(c):
            w = float(g["os_event"].sum())
            num, den = num + w * c, den + w
    return num / den if den else float("nan")


def fit_pool_cox(features, lab, penalizer=0.05):
    # Cox on pooled training cohorts, stratified by cohort (each cohort keeps its own baseline hazard).
    df = features.astype(float).assign(
        duration=lab.loc[features.index, "os_time_days"].values,
        status=lab.loc[features.index, "os_event"].values,
        stratum=lab.loc[features.index, "cohort"].values,
    )
    cph = CoxPHFitter(penalizer=penalizer)
    cph.fit(df, duration_col="duration", event_col="status", strata=["stratum"], show_progress=False)
    return cph


def oof_pool_cox(features, lab, penalizer=0.05, seed=SEED):
    # 5-fold OOF linear predictor of the stratified pool Cox (folds stratified by cohort x event).
    event = lab.loc[features.index, "os_event"].astype(int)
    groups = lab.loc[features.index, "cohort"].astype(str) + "_" + event.astype(str)
    lp = pd.Series(index=features.index, dtype=float)
    for tr, te in StratifiedKFold(5, shuffle=True, random_state=seed).split(features, groups):
        cph = fit_pool_cox(features.iloc[tr], lab, penalizer)
        lp.iloc[te] = linear_predictor(cph, features.iloc[te])
    return lp


def oof_cox(features, time, event, penalizer=0.0, l1_ratio=0.0, seed=SEED):
    # Out-of-fold linear predictor. Higher = higher risk.
    event = pd.Series(event, index=features.index).astype(int)
    time = pd.Series(time, index=features.index)
    n_pos, n_neg = int(event.sum()), int((event == 0).sum())
    n_splits = int(min(5, n_pos, n_neg, max(len(features) // 15, 2)))
    n_splits = max(n_splits, 2)
    risk = pd.Series(index=features.index, dtype=float)
    last = None
    if n_pos < 5 or len(features) < 20:
        df = features.assign(duration=time.values, status=event.values)
        last = CoxPHFitter(penalizer=max(penalizer, 0.01), l1_ratio=l1_ratio)
        last.fit(df, duration_col="duration", event_col="status", show_progress=False)
        risk[:] = linear_predictor(last, features)
        return risk, last, "in_sample"
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    for tr, te in cv.split(features, event):
        Xtr, Xte = features.iloc[tr], features.iloc[te]
        df = Xtr.assign(duration=time.iloc[tr].values, status=event.iloc[tr].values)
        cph = CoxPHFitter(penalizer=penalizer, l1_ratio=l1_ratio)
        cph.fit(df, duration_col="duration", event_col="status", show_progress=False)
        risk.iloc[te] = linear_predictor(cph, Xte)
        last = cph
    return risk, last, f"{n_splits}-fold"

## 6. M3 — clinical Cox on every cohort

Complete-case, cohort-available covariates among residual / stage / age.
Columns with fewer than 2 distinct values are dropped (stage still keeps III vs IV
when both exist). 5-fold OOF C-index when the cohort is large enough; otherwise
in-sample (flagged).

A **transported** residual+stage Cox is also fit on the training-pool complete
cases (stratified by training cohort) and applied to each validator — same train/score protocol as the expression
models, used for ΔC.

This is the first cell that touches validation outcomes, so (unless this is a dry run) it
registers the frozen candidate in `.research/validation-ledger.jsonl` first (`ce.register_validation`).
Re-running the notebook after a successful save is refused: tune on training-pool
CV, and give a `reason` (disclosed in the write-up) if a re-score is legitimate.

In [ ]:
# Frozen first-pass candidate: every choice that can move validation numbers.
VALIDATION_CANDIDATE = "os-first-pass-v1"
VALIDATION_CONFIG = {
    "n_genes": N_GENES, "seed": SEED, "n_boot": N_BOOT, "clinical_covars": CLIN_COLS,
    "expression": "within-sample rank pct; top-var genes on pool; z-score on pool; val NA -> 0.5",
    "clinical": {"per_cohort_penalizer": 0.05, "transported": ["residual_subopt", "stage_ord"]},
    "lasso_cox": {"engine": "sksurv Coxnet l1_ratio=1", "alpha_path": "alpha_min_ratio=0.05, n_alphas=12",
                  "selection": "train OOF C-index"},
    "rsf": {"n_estimators": 200, "min_samples_split": 10, "min_samples_leaf": 5, "max_features": "sqrt"},
    "deepsurv": {"hidden": 32, "dropout": 0.3, "epochs": 80, "lr": 1e-3, "weight_decay": 1e-4},
    "pool_cox_strata": "training cohort",
    "expr_clin_cox": {"primary": True, "stage1": "lasso_cox linear predictor, OOF on pool, z-scored on pool",
                      "stage2": ["residual_subopt", "stage_ord", "expr_score"], "penalizer": 0.05},
    "delta_ci": "paired bootstrap", "pooled": "DerSimonian-Laird over validators",
}
if DRY_RUN:
    print(f"dry run: {VALIDATION_CANDIDATE} NOT registered")
else:
    ce.register_validation(VALIDATION_CANDIDATE, VAL_COHORTS, config=VALIDATION_CONFIG)


def available_covars(frame, min_n=20):
    cols = []
    for c in CLIN_COLS:
        s = frame[c]
        if s.notna().sum() >= min_n and s.nunique(dropna=True) >= 2:
            cols.append(c)
    return cols


def fit_clinical_cohort(lab, min_n=20):
    cols = available_covars(lab, min_n=min_n)
    if not cols:
        return None
    mask = lab[cols].notna().all(axis=1)
    sub = lab.loc[mask]
    if len(sub) < 20 or int(sub["os_event"].sum()) < 5:
        return None
    feat = sub[cols].astype(float)
    risk, model, how = oof_cox(feat, sub["os_time_days"], sub["os_event"], penalizer=0.05)
    metrics = pack_metrics(sub["os_time_days"], sub["os_event"], risk)
    metrics["events"] = int(sub["os_event"].sum())
    metrics["n"] = int(len(sub))
    metrics["covars"] = cols
    metrics["how"] = how
    return {"metrics": metrics, "risk": risk, "model": model, "index": sub.index, "covars": cols}


clinical_rows = []
clinical_by_cohort = {}
for split, lab in ("train", lab_tr), ("val", lab_va):
    for cohort, g in lab.groupby("cohort"):
        fit = fit_clinical_cohort(g)
        clinical_by_cohort[cohort] = fit
        if fit is None:
            clinical_rows.append({"split": split, "cohort": cohort, "n_lab": len(g),
                                  "status": "skipped (no usable covariates)"})
            continue
        m = fit["metrics"]
        clinical_rows.append({
            "split": split, "cohort": cohort, "n_lab": len(g),
            "n_model": m["n"], "events": m["events"],
            "covars": "+".join(fit["covars"]), "how": m["how"],
            "cindex": m["cindex"], "ci_lo": m["cindex_ci_lo"], "ci_hi": m["cindex_ci_hi"],
            "logrank_p": m["logrank_p"],
        })

clin_table = pd.DataFrame(clinical_rows)
print("Per-cohort clinical Cox")
print(clin_table.to_string(index=False, float_format=lambda x: f"{x:.3f}"))

pool_cols = ["residual_subopt", "stage_ord"]
pool_mask = lab_tr[pool_cols].notna().all(axis=1)
pool_feat = lab_tr.loc[pool_mask, pool_cols].astype(float)
print(f"\ntransported clinical train n={len(pool_feat)} events={int(lab_tr.loc[pool_mask, 'os_event'].sum())}")
print(lab_tr.loc[pool_mask, "cohort"].value_counts().to_string())
pool_cph = fit_pool_cox(pool_feat, lab_tr, penalizer=0.05)
print(pool_cph.summary[["coef", "exp(coef)", "p"]].round(3))

transported = {}
for cohort in VAL_COHORTS:
    g = lab_va.loc[lab_va["cohort"] == cohort]
    msk = g[pool_cols].notna().all(axis=1)
    sub = g.loc[msk]
    risk = linear_predictor(pool_cph, sub[pool_cols])
    risk = pd.Series(risk, index=sub.index)
    met = pack_metrics(sub["os_time_days"], sub["os_event"], risk)
    met["events"] = int(sub["os_event"].sum())
    met["n"] = int(len(sub))
    transported[cohort] = {"metrics": met, "risk": risk, "index": sub.index}
    print(f"  transported {cohort:12s} n={met['n']:3d}  C={met['cindex']:.3f}  "
          f"logrank p={met['logrank_p']}")

## 7. Expression models and the primary expression + clinical model

Same 500 rank-z-scored genes for all three classes. The LASSO alpha is chosen by training
OOF C-index (not a result). RSF and DeepSurv use fixed first-pass
defaults; their training C-index is an overfit check.

**Primary model `expr_clin_cox` (D009).** Stage 1 is the LASSO-Cox expression score. Stage 2 is
a Cox on residual + stage + that score, fit on the pool's residual+stage complete cases and
stratified by cohort, i.e. exactly the transported clinical model plus one expression term, so
ΔC vs `clinical_transported` isolates what expression adds. Stage 2 learns from the
**out-of-fold** score (an in-sample score would inflate its weight). Pool OOF C of both models
is an internal check only.

In [ ]:
from sksurv.ensemble import RandomSurvivalForest
from sksurv.linear_model import CoxnetSurvivalAnalysis
from sksurv.util import Surv

y_tr = Surv.from_arrays(event_tr.to_numpy().astype(bool), time_tr.to_numpy())


# LASSO-Cox by coordinate descent (Coxnet): lifelines' L1 Cox does not scale to 500 covariates.
# The alpha path is fixed on the pool, then scored by out-of-fold C-index. Only the alpha comes
# out of this — a training-CV choice; no validation data is involved.
def coxnet(alphas=None, **kw):
    return CoxnetSurvivalAnalysis(l1_ratio=1.0, alphas=alphas, max_iter=100_000, tol=1e-7,
                                  fit_baseline_model=False, **kw)


ALPHAS = [float(a) for a in coxnet(alpha_min_ratio=0.05, n_alphas=12)
          .fit(Z_tr.to_numpy(), y_tr).alphas_]
print(f"Coxnet alpha path {ALPHAS[0]:.4f} -> {ALPHAS[-1]:.4f} ({len(ALPHAS)} values)")

oof_lp = {a: pd.Series(index=Z_tr.index, dtype=float) for a in ALPHAS}
for tr, te in StratifiedKFold(5, shuffle=True, random_state=SEED).split(Z_tr, event_tr):
    fold = coxnet(alphas=ALPHAS).fit(Z_tr.iloc[tr].to_numpy(), y_tr[tr])
    for a in ALPHAS:
        oof_lp[a].iloc[te] = fold.predict(Z_tr.iloc[te].to_numpy(), alpha=a)

print("LASSO-Cox alpha grid (training OOF)")
lasso_grid = [{"alpha": a, "oof_cindex": cindex(time_tr, event_tr, oof_lp[a]), "risk": oof_lp[a]}
              for a in ALPHAS]
for d in lasso_grid:
    print(f"  alpha={d['alpha']:.4f}  OOF C-index={d['oof_cindex']:.3f}")
best_lasso = max(lasso_grid, key=lambda d: d["oof_cindex"])
print(f"chosen alpha={best_lasso['alpha']:.4f}  OOF C={best_lasso['oof_cindex']:.3f}")

lasso_full = coxnet(alphas=ALPHAS).fit(Z_tr.to_numpy(), y_tr)
lasso_coef = pd.Series(lasso_full.coef_[:, ALPHAS.index(best_lasso["alpha"])], index=Z_tr.columns)
sig = lasso_coef[lasso_coef.abs() > 1e-8].sort_values(key=lambda s: s.abs(), ascending=False)
print(f"LASSO signature {len(sig)} / {N_GENES} genes  top:")
print(sig.head(15).round(4).to_string())


def lasso_lp(Z):
    return Z[lasso_coef.index].to_numpy() @ lasso_coef.to_numpy()


print("\nRandom survival forest (200 trees, sqrt features)")
rsf = RandomSurvivalForest(
    n_estimators=200,
    min_samples_split=10,
    min_samples_leaf=5,
    max_features="sqrt",
    n_jobs=-1,
    random_state=SEED,
)
rsf.fit(Z_tr.to_numpy(), y_tr)
rsf_tr_risk = rsf.predict(Z_tr.to_numpy())
print(f"  train (in-sample) C-index={cindex(time_tr, event_tr, rsf_tr_risk):.3f}  "
      "[overfit check, not evidence]")

import torch
import torch.nn as nn


class DeepSurvNet(nn.Module):
    def __init__(self, n_in, hidden=32, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_in, hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)


def cox_ph_loss(risk, time, event):
    order = torch.argsort(time, descending=True)
    risk, event = risk[order], event[order]
    log_cum = torch.logcumsumexp(risk, dim=0)
    denom = event.sum().clamp(min=1.0)
    return -((risk - log_cum) * event).sum() / denom


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(SEED)
deep = DeepSurvNet(Z_tr.shape[1]).to(device)
xt = torch.tensor(Z_tr.to_numpy(), dtype=torch.float32, device=device)
tt = torch.tensor(time_tr.to_numpy(), dtype=torch.float32, device=device)
et = torch.tensor(event_tr.to_numpy(), dtype=torch.float32, device=device)
opt = torch.optim.Adam(deep.parameters(), lr=1e-3, weight_decay=1e-4)
print(f"\nDeepSurv on {device}  {Z_tr.shape[1]}->32->1")
deep.train()
for epoch in range(1, 81):
    opt.zero_grad()
    loss = cox_ph_loss(deep(xt), tt, et)
    loss.backward()
    opt.step()
    if epoch in (20, 40, 60, 80):
        print(f"  epoch {epoch:3d}  loss={loss.item():.4f}")
deep.eval()
with torch.no_grad():
    deep_tr_risk = deep(xt).cpu().numpy()
print(f"  train (in-sample) C-index={cindex(time_tr, event_tr, deep_tr_risk):.3f}  "
      "[overfit check, not evidence]")


def predict_deep(model, Z):
    model.eval()
    with torch.no_grad():
        t = torch.tensor(Z.to_numpy(), dtype=torch.float32, device=device)
        return model(t).cpu().numpy()


expr_models = {
    "lasso_cox": {
        "predict": lambda Z, lab: lasso_lp(Z),
        "train_oof_cindex": float(best_lasso["oof_cindex"]),
        "alpha": float(best_lasso["alpha"]),
        "alpha_path": ALPHAS,
        "alpha_grid_oof_cindex": {f"{d['alpha']:.5f}": d["oof_cindex"] for d in lasso_grid},
        "signature_size": int(len(sig)),
        "signature_top": {k: float(v) for k, v in sig.head(20).items()},
    },
    "rsf": {
        "predict": lambda Z, lab: rsf.predict(Z.to_numpy()),
        "train_insample_cindex": float(cindex(time_tr, event_tr, rsf_tr_risk)),
        "n_estimators": 200,
    },
    "deepsurv": {
        "predict": lambda Z, lab: predict_deep(deep, Z),
        "train_insample_cindex": float(cindex(time_tr, event_tr, deep_tr_risk)),
        "hidden": 32, "epochs": 80, "device": str(device),
    },
}

# ---- primary: expression + clinical (two-stage) ----
expr_oof = best_lasso["risk"]
expr_oof_z = (expr_oof - expr_oof.mean()) / expr_oof.std(ddof=0)
lasso_pool_lp = lasso_lp(Z_tr)
LASSO_MU, LASSO_SD = float(lasso_pool_lp.mean()), float(lasso_pool_lp.std(ddof=0))
if LASSO_SD < 1e-12:
    raise SystemExit("LASSO-Cox selected no genes: expression score is constant")


def expr_score(Z):
    return pd.Series((lasso_lp(Z) - LASSO_MU) / LASSO_SD, index=Z.index)


comb_feat = pool_feat.assign(expr_score=expr_oof_z.loc[pool_feat.index].values)
comb_cph = fit_pool_cox(comb_feat, lab_tr, penalizer=0.05)
print(f"\nexpr_clin_cox stage 2 on pool n={len(comb_feat)} (stratified by cohort)")
print(comb_cph.summary[["coef", "exp(coef)", "exp(coef) lower 95%", "exp(coef) upper 95%", "p"]].round(3))


def predict_combined(Z, lab):
    feat = lab.loc[Z.index, pool_cols].astype(float).assign(expr_score=expr_score(Z).values)
    ok = feat.notna().all(axis=1).to_numpy()
    out = np.full(len(feat), np.nan)
    out[ok] = linear_predictor(comb_cph, feat.loc[ok])
    return out


pool_sub = lab_tr.loc[pool_feat.index]
clin_pool_oof = oof_pool_cox(pool_feat, lab_tr)
comb_pool_oof = oof_pool_cox(comb_feat, lab_tr)
pool_internal = {
    "n": int(len(pool_sub)), "events": int(pool_sub["os_event"].sum()),
    "clinical_transported_oof_cindex": stratified_cindex(pool_sub, clin_pool_oof),
    "expr_clin_cox_oof_cindex": stratified_cindex(pool_sub, comb_pool_oof),
    "lasso_cox_oof_cindex_same_patients": stratified_cindex(pool_sub, expr_oof.loc[pool_sub.index]),
}
print("pool internal check (cohort-stratified OOF C, not evidence):",
      {k: round(v, 3) if isinstance(v, float) else v for k, v in pool_internal.items()})

expr_models = {
    "expr_clin_cox": {
        "predict": predict_combined,
        "primary": True,
        "stage2_coef": {k: float(v) for k, v in comb_cph.params_.items()},
        "stage2_hr_ci": comb_cph.summary[["exp(coef)", "exp(coef) lower 95%", "exp(coef) upper 95%", "p"]]
                        .round(4).to_dict(orient="index"),
        "lasso_score_mu_sd": [LASSO_MU, LASSO_SD],
        "pool_internal": pool_internal,
    },
    **expr_models,
}

## 8. External validation — all five cohorts, all classes

For each validator: C-index with bootstrap 95% CI, KM log-rank (median split),
and ΔC vs (a) that cohort's own clinical OOF and (b) the transported
residual+stage Cox, with a paired bootstrap 95% CI. Expression ΔC vs clinical uses the
**clinical complete-case subset** so the comparison is on the same patients.

Then per model a random-effects pooled ΔC over the validators, and per validator the
LASSO expression score's hazard ratio (per SD) adjusted for residual + stage, with a
likelihood-ratio test for adding it (refit within the validator; descriptive).

In [ ]:
def eval_risk(lab, risk):
    return pack_metrics(lab["os_time_days"], lab["os_event"], risk)


external = {}
rows = []
for cohort in VAL_COHORTS:
    g = lab_va.loc[lab_va["cohort"] == cohort]
    Zc = Z_va.loc[g.index]
    rec = {
        "n": int(len(g)),
        "events": int(g["os_event"].sum()),
        "platform": str(g["platform"].iloc[0]),
    }
    clin_fit = clinical_by_cohort.get(cohort)
    trans = transported.get(cohort)

    if clin_fit is not None:
        rec["clinical_oof"] = clin_fit["metrics"]
    if trans is not None:
        rec["clinical_transported"] = trans["metrics"]

    for name, spec in expr_models.items():
        risk = pd.Series(spec["predict"](Zc, g), index=g.index)
        full = eval_risk(g, risk)
        rec[name] = dict(full)
        delta_oof = None
        if clin_fit is not None:
            idx = clin_fit["index"]
            expr_on = pack_metrics(lab_va.loc[idx, "os_time_days"],
                                   lab_va.loc[idx, "os_event"], risk.loc[idx])
            rec[name]["cindex_on_clinical_subset"] = expr_on["cindex"]
            if np.isfinite(expr_on["cindex"]) and np.isfinite(clin_fit["metrics"]["cindex"]):
                delta_oof = expr_on["cindex"] - clin_fit["metrics"]["cindex"]
            rec[name]["delta_c_vs_clinical_oof"] = delta_oof
        delta_tr = d_lo = d_hi = None
        if trans is not None:
            idx = trans["index"]
            delta_tr, d_lo, d_hi, d_se = delta_ci(lab_va.loc[idx, "os_time_days"], lab_va.loc[idx, "os_event"],
                                                  risk.loc[idx], trans["risk"])
            rec[name]["delta_c_vs_clinical_transported"] = delta_tr
            rec[name]["delta_c_vs_clinical_transported_ci"] = [d_lo, d_hi]
            rec[name]["delta_c_vs_clinical_transported_se"] = d_se
        rec[name]["risk"] = risk
        rows.append({
            "cohort": cohort, "n": rec["n"], "events": rec["events"],
            "model": name,
            "cindex": full["cindex"], "ci_lo": full["cindex_ci_lo"], "ci_hi": full["cindex_ci_hi"],
            "logrank_p": full["logrank_p"],
            "delta_vs_clin_oof": delta_oof,
            "delta_vs_clin_transported": delta_tr, "d_tr_lo": d_lo, "d_tr_hi": d_hi,
        })
    if clin_fit is not None:
        m = clin_fit["metrics"]
        rows.append({
            "cohort": cohort, "n": m["n"], "events": m["events"],
            "model": "clinical_oof(" + "+".join(clin_fit["covars"]) + ")",
            "cindex": m["cindex"], "ci_lo": m["cindex_ci_lo"], "ci_hi": m["cindex_ci_hi"],
            "logrank_p": m["logrank_p"],
            "delta_vs_clin_oof": 0.0, "delta_vs_clin_transported": None,
        })
    if trans is not None:
        m = trans["metrics"]
        rows.append({
            "cohort": cohort, "n": m["n"], "events": m["events"],
            "model": "clinical_transported(rd+stage)",
            "cindex": m["cindex"], "ci_lo": m["cindex_ci_lo"], "ci_hi": m["cindex_ci_hi"],
            "logrank_p": m["logrank_p"],
            "delta_vs_clin_oof": None, "delta_vs_clin_transported": 0.0,
        })
    if trans is not None:
        sub = lab_va.loc[trans["index"]]
        cols = [c for c in pool_cols if sub[c].nunique() >= 2]
        base = sub[cols].astype(float).assign(duration=sub["os_time_days"].values,
                                              status=sub["os_event"].values)
        full = base.assign(expr_score=expr_score(Zc.loc[sub.index]).values)
        m0 = CoxPHFitter().fit(base, duration_col="duration", event_col="status")
        m1 = CoxPHFitter().fit(full, duration_col="duration", event_col="status")
        s = m1.summary.loc["expr_score"]
        lr = 2.0 * (m1.log_likelihood_ - m0.log_likelihood_)
        rec["expr_score_added_value"] = {
            "n": int(len(sub)), "adjusted_for": cols,
            "hr_per_sd": float(s["exp(coef)"]), "hr_lo": float(s["exp(coef) lower 95%"]),
            "hr_hi": float(s["exp(coef) upper 95%"]), "wald_p": float(s["p"]),
            "lr_stat": float(lr), "lr_p": float(chi2.sf(max(lr, 0.0), 1)),
        }
    external[cohort] = rec

ext_table = pd.DataFrame(rows)
print("External validation (all cohorts x all classes)")
print(ext_table.to_string(index=False, float_format=lambda x: f"{x:.3f}"))

pooled = {}
for name in expr_models:
    pooled[name] = random_effects(
        [external[c][name].get("delta_c_vs_clinical_transported") for c in VAL_COHORTS],
        [external[c][name].get("delta_c_vs_clinical_transported_se") for c in VAL_COHORTS])
print("\nRandom-effects pooled Delta C vs transported clinical")
for name, p in pooled.items():
    if p:
        print(f"  {name:14s} {p['estimate']:+.3f} [{p['ci_lo']:+.3f}, {p['ci_hi']:+.3f}]  I2={p['i2']:.2f}  k={p['k']}")

print("\nLASSO expression score adjusted for residual + stage (within validator)")
for c in VAL_COHORTS:
    av = external[c].get("expr_score_added_value")
    if av:
        print(f"  {c:10s} n={av['n']:3d}  HR/SD={av['hr_per_sd']:.2f} [{av['hr_lo']:.2f}, {av['hr_hi']:.2f}]  LR p={av['lr_p']:.3g}")

print("\nPrimary: expr_clin_cox Delta C >= 0.03 vs clinical_transported on >= 2 validators (D006, D009).")
print("GSE140082 is immature (max OS ~3.6 y). GSE49997 is tertiary / short follow-up.")

## 9. Kaplan–Meier on the flagship (GSE32062)

Median split of predicted risk for each class, clinical baselines included. Other validators are in the table.

In [ ]:
import matplotlib.pyplot as plt

flag = "gse32062"
g = lab_va.loc[lab_va["cohort"] == flag]
panels = [("clinical transported", transported[flag]["risk"], transported[flag]["index"])]
if clinical_by_cohort.get(flag) is not None:
    panels.append(("clinical OOF", clinical_by_cohort[flag]["risk"],
                   clinical_by_cohort[flag]["index"]))
for name in ("expr_clin_cox", "lasso_cox", "rsf", "deepsurv"):
    panels.append((name, external[flag][name]["risk"], g.index))

fig, axes = plt.subplots(2, 3, figsize=(15, 8), sharey=True)
axes = axes.ravel()
for ax, (title, risk, idx) in zip(axes, panels):
    sub = lab_va.loc[idx]
    r = pd.Series(np.asarray(risk, float), index=idx)
    r = r[np.isfinite(r)]
    sub = sub.loc[r.index]
    hi = r >= r.median()
    kmf = KaplanMeierFitter()
    kmf.fit(sub.loc[hi, "os_time_days"], sub.loc[hi, "os_event"], label="high risk")
    kmf.plot_survival_function(ax=ax, ci_show=False)
    kmf.fit(sub.loc[~hi, "os_time_days"], sub.loc[~hi, "os_event"], label="low risk")
    kmf.plot_survival_function(ax=ax, ci_show=False)
    p = km_logrank(sub["os_time_days"], sub["os_event"], r)
    ci = cindex(sub["os_time_days"], sub["os_event"], r)
    ax.set_title(f"{title}\nC={ci:.3f}  log-rank p={p:.3g}" if p is not None else title)
    ax.set_xlabel("days")
    ax.set_ylabel("OS")
fig.suptitle("GSE32062 flagship — first-pass models" + ("  [DRY RUN: synthetic outcomes]" if DRY_RUN else ""))
fig.tight_layout()
import tempfile
KM_PATH = pathlib.Path(tempfile.gettempdir()) / "km-gse32062-first-pass.png"
fig.savefig(KM_PATH, dpi=120)
plt.show()

## 10. Save the run

Writes `.research/colab/runs/<utc>-os-first-pass/run.json` on the runtime and
prints it between markers. `colab_sync.py run --experiment E003` pulls the record
back and links it to E003. Metric rows (`ce.metric_row`) are what the tracker
queries: `research.py results --experiment E003`, and `research.py verdict E003`
applies the pre-registered success rule to `delta_c_vs_clinical_transported` of the primary
model `expr_clin_cox`. A dry run validates the rows and saves nothing.

In [ ]:
def strip_risk(obj):
    if isinstance(obj, dict):
        return {k: strip_risk(v) for k, v in obj.items() if k != "risk"}
    return obj


payload = {
    "task": "os first-pass M3 clinical + M4 expression classes",
    "preprocess": {
        "expression": "within-sample rank percentile; top 500 by train variance; z-score on train; val NA -> 0.5",
        "residual": "GOG optimal vs suboptimal (os_clinical.recode_residual)",
        "stage": "FIGO 1-4 (os_clinical.recode_stage)",
        "imputation": "none for clinical; rank mid-fill on validation coverage gaps",
        "n_genes": N_GENES,
    },
    "train": {
        "n": int(len(lab_tr)),
        "events": int(event_tr.sum()),
        "n_features_total": int(X_tr.shape[1]),
        "n_features_model": N_GENES,
        "n_clinical_rd_stage": int(pool_mask.sum()),
    },
    "clinical_per_cohort": clin_table.to_dict(orient="records"),
    "models": {
        "lasso_cox": {k: v for k, v in expr_models["lasso_cox"].items() if k != "predict"},
        "rsf": {k: v for k, v in expr_models["rsf"].items() if k != "predict"},
        "deepsurv": {k: v for k, v in expr_models["deepsurv"].items() if k != "predict"},
        "expr_clin_cox": {k: v for k, v in expr_models["expr_clin_cox"].items() if k != "predict"},
    },
    "primary_model": "expr_clin_cox",
    "validation_config": VALIDATION_CONFIG,
    "pooled_delta_c_vs_clinical_transported": pooled,
    "external": {c: strip_risk(rec) for c, rec in external.items()},
}

rows = []
for _, r in clin_table.iterrows():  # per-cohort clinical Cox (CV within each cohort)
    if pd.notna(r.get("cindex")):
        rows.append(ce.metric_row("clinical_oof", r["cohort"], "cv", "cindex", r["cindex"],
                                  ci_lo=r.get("ci_lo"), ci_hi=r.get("ci_hi"), n=int(r["n_model"])))
rows.append(ce.metric_row("lasso_cox", "os-training-pool", "cv", "cindex", best_lasso["oof_cindex"],
                          n=int(len(lab_tr))))
rows.append(ce.metric_row("clinical_transported", "os-training-pool", "cv", "cindex",
                          pool_internal["clinical_transported_oof_cindex"], n=pool_internal["n"]))
rows.append(ce.metric_row("expr_clin_cox", "os-training-pool", "cv", "cindex",
                          pool_internal["expr_clin_cox_oof_cindex"], n=pool_internal["n"]))
rows.append(ce.metric_row("rsf", "os-training-pool", "train", "cindex", expr_models["rsf"]["train_insample_cindex"]))
rows.append(ce.metric_row("deepsurv", "os-training-pool", "train", "cindex",
                          expr_models["deepsurv"]["train_insample_cindex"]))
for cohort, rec in external.items():
    if "clinical_transported" in rec:
        m = rec["clinical_transported"]
        rows.append(ce.metric_row("clinical_transported", cohort, "external", "cindex", m["cindex"],
                                  ci_lo=m["cindex_ci_lo"], ci_hi=m["cindex_ci_hi"], n=m["n"]))
        rows.append(ce.metric_row("clinical_transported", cohort, "external", "logrank_p", m["logrank_p"], n=m["n"]))
    for name in expr_models:
        m = rec[name]
        rows.append(ce.metric_row(name, cohort, "external", "cindex", m["cindex"],
                                  ci_lo=m["cindex_ci_lo"], ci_hi=m["cindex_ci_hi"], n=m["n"]))
        rows.append(ce.metric_row(name, cohort, "external", "logrank_p", m["logrank_p"], n=m["n"]))
        d_lo, d_hi = m.get("delta_c_vs_clinical_transported_ci") or (None, None)
        rows.append(ce.metric_row(name, cohort, "external", "delta_c_vs_clinical_transported",
                                  m.get("delta_c_vs_clinical_transported"), ci_lo=d_lo, ci_hi=d_hi,
                                  baseline="clinical_transported"))
        rows.append(ce.metric_row(name, cohort, "external", "delta_c_vs_clinical_oof",
                                  m.get("delta_c_vs_clinical_oof"), baseline="clinical_oof"))

    av = rec.get("expr_score_added_value")
    if av:
        rows.append(ce.metric_row("lasso_cox", cohort, "external", "hr_per_sd_adj_rd_stage", av["hr_per_sd"],
                                  ci_lo=av["hr_lo"], ci_hi=av["hr_hi"], n=av["n"]))
        rows.append(ce.metric_row("lasso_cox", cohort, "external", "lr_p_added_to_rd_stage", av["lr_p"], n=av["n"]))
for name, p in pooled.items():
    if p:
        rows.append(ce.metric_row(name, "os-validation", "external", "delta_c_vs_clinical_transported_pooled",
                                  p["estimate"], ci_lo=p["ci_lo"], ci_hi=p["ci_hi"], n=p["k"],
                                  baseline="clinical_transported"))

if DRY_RUN:
    problems = ce._check_metric_rows([r for r in rows if r is not None])
    print(f"DRY RUN: {sum(r is not None for r in rows)} metric rows built, problems={problems}; nothing saved")
    raise SystemExit(1 if problems else 0)

run_dir = ce.save_run("os-first-pass", payload, metrics=rows, files=[str(KM_PATH)])
print(f"{sum(r is not None for r in rows)} metric rows saved")
print(run_dir)
print("===RUN-RECORD-BEGIN===")
print((run_dir / "run.json").read_text().strip())
print("===RUN-RECORD-END===")